# 04 — Validación Gold → PostgreSQL

El notebook utiliza el mismo punto de entrada `ensure_serving_ready()` que Docker. Migra automáticamente un esquema `analytics` antiguo e incompatible y luego realiza una carga incremental e idempotente.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Raíz del proyecto:', PROJECT_ROOT)

In [ ]:
from config.settings import get_postgres_settings
from serving.postgres_loader import ensure_serving_ready, fetch_serving_metrics, read_gold
import psycopg

GOLD_ROOT = PROJECT_ROOT/'data'/'gold'
SCHEMA_SQL = PROJECT_ROOT/'docker'/'postgres'/'init'/'001_schema.sql'
settings = get_postgres_settings()
print(f'{settings.host}:{settings.port}/{settings.database} schema={settings.schema}')

In [ ]:
serving_summary = ensure_serving_ready(
    gold_root=GOLD_ROOT,
    schema_sql_path=SCHEMA_SQL,
    settings=settings,
)
display(pd.DataFrame([serving_summary]))
print('El serving PostgreSQL está listo.')

## Reconciliación Gold ↔ PostgreSQL

In [ ]:
gold = read_gold(GOLD_ROOT)
metrics = fetch_serving_metrics(settings)
expected = {
    'dim_process': len(gold['dim_process']),
    'dim_date': len(gold['dim_date']),
    'fact_process_daily': len(gold['fact_process_daily']),
    'gold_event_count': int(gold['fact_process_daily']['event_count'].sum()),
}
reconciliation = pd.DataFrame({
    'metric': list(expected),
    'gold_value': list(expected.values()),
    'postgres_value': [metrics[k] for k in expected],
})
reconciliation['matches'] = reconciliation['gold_value'].eq(reconciliation['postgres_value'])
display(reconciliation)
assert reconciliation['matches'].all()

## Estado incremental

In [ ]:
with psycopg.connect(**settings.connection_kwargs) as connection:
    load_state = pd.read_sql_query(
        f'''SELECT dataset, source_path, row_count, event_count, loaded_at
            FROM {settings.schema}.etl_load_state
            ORDER BY dataset, source_path''',
        connection,
    )
display(load_state)

## Contrato de la vista de serving

In [ ]:
with psycopg.connect(**settings.connection_kwargs) as connection:
    preview = pd.read_sql_query(
        f'''SELECT local_date, factory_name, line_name, work_area_name,
                   tightening_unit_name, process_step_name, substep_number,
                   event_count, torque_applicable_rate_pct, torque_in_spec_rate_pct,
                   torque_out_of_spec_rate_pct
            FROM {settings.schema}.vw_process_daily_kpis
            ORDER BY local_date, line_name, tightening_unit_name
            LIMIT 20''',
        connection,
    )
display(preview)

## Comprobación de idempotencia

Ejecutar nuevamente `ensure_serving_ready()` sin nuevas particiones Gold no debe cargar nada y debe omitir las particiones ya registradas.

In [ ]:
second_run = ensure_serving_ready(GOLD_ROOT, SCHEMA_SQL, settings)
display(pd.DataFrame([second_run]))
assert second_run['loaded_fact_partitions'] == []
print('Comprobación de serving incremental/idempotente superada.')